In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fnnds-nutritional-estimation-final-25-04/fnnds_selected_columns_dataset.csv
/kaggle/input/nutitional-estimation/selected_columns_dataset.csv


In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [27]:
data = pd.read_csv('/kaggle/input/fnnds-nutritional-estimation-final-25-04/fnnds_selected_columns_dataset.csv')

In [28]:
# Ensure the columns are clean
data.columns = [col.strip().replace("\n", " ").replace(" ", "_").lower() for col in data.columns]

In [29]:
# Define features and target columns
features = ['main_food_description']
targets = ['energy_(kcal)', 'protein_(g)', 'carbohydrate_(g)', 
            'sugars,_total_(g)', 'fiber,_total_dietary_(g)', 'total_fat_(g)', 
            'fatty_acids,_total_saturated_(g)',
            'fatty_acids,_total_monounsaturated_(g)',
            'fatty_acids,_total_polyunsaturated_(g)', 'cholesterol_(mg)']


In [30]:
X = data[features]
y = data[targets]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), features)
    ])

In [32]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_estimators=100))
])

In [33]:
# Train the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['main_food_description'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [34]:
y_pred = pipeline.predict(X_test)
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))

Mean Squared Error: 4771.9132767838455


In [36]:
# Function to predict nutrient values
def predict_nutrients(food_description, quantity_in_grams):
    input_data = pd.DataFrame({
        'main_food_description': [food_description]
    })

    # Predict nutrient values per 100g
    nutrient_values_100g = pipeline.predict(input_data)[0]

    # Scale predictions for the given quantity
    nutrient_values_scaled = (nutrient_values_100g * quantity_in_grams) / 100

    # Create a dictionary for the results
    nutrients = {
        'energy_(kcal)': nutrient_values_scaled[0],
        'protein_(g)': nutrient_values_scaled[1],
        'carbohydrate_(g)': nutrient_values_scaled[2],
        'sugars_total_(g)': nutrient_values_scaled[3],
        'fiber_total_dietary_(g)': nutrient_values_scaled[4],
        'total_fat_(g)': nutrient_values_scaled[5],
        'fatty_acids,_total_saturated_(g)': nutrient_values_scaled[6],
        'fatty_acids,_total_monounsaturated_(g)': nutrient_values_scaled[7],
        'fatty_acids,_total_polyunsaturated_(g)': nutrient_values_scaled[8],
        'cholesterol_(mg)': nutrient_values_scaled[9]
    }

    return nutrients

In [37]:
test_cases = [
    {"food": "Milk, whole", "quantity": 250},
    {"food": "Bread, whole wheat", "quantity": 100},
    {"food": "Apple, raw", "quantity": 150},
    {"food": "Cheese, cheddar", "quantity": 50},
    {"food": "Chicken, roasted", "quantity": 200},
    {"food": "Rice, white, cooked", "quantity": 180},
    {"food": "Broccoli, steamed", "quantity": 100},
    {"food": "Egg, boiled", "quantity": 70},
    {"food": "Butter, salted", "quantity": 20},
    {"food": "Banana, raw", "quantity": 120}
]


In [38]:
for case in test_cases:
    food = case["food"]
    quantity = case["quantity"]
    predicted_nutrients = predict_nutrients(food, quantity)
    
    print(f"\nPredicted nutrients for {quantity}g of {food}:")
    for nutrient, value in predicted_nutrients.items():
        print(f"{nutrient}: {value:.2f}")


Predicted nutrients for 250g of Milk, whole:
energy_(kcal): 135.90
protein_(g): 5.45
carbohydrate_(g): 15.64
sugars_total_(g): 14.34
fiber_total_dietary_(g): 0.67
total_fat_(g): 5.78
fatty_acids,_total_saturated_(g): 2.96
fatty_acids,_total_monounsaturated_(g): 1.37
fatty_acids,_total_polyunsaturated_(g): 0.49
cholesterol_(mg): 17.55

Predicted nutrients for 100g of Bread, whole wheat:
energy_(kcal): 192.12
protein_(g): 8.87
carbohydrate_(g): 32.69
sugars_total_(g): 5.20
fiber_total_dietary_(g): 4.36
total_fat_(g): 2.88
fatty_acids,_total_saturated_(g): 0.64
fatty_acids,_total_monounsaturated_(g): 0.56
fatty_acids,_total_polyunsaturated_(g): 1.22
cholesterol_(mg): 0.19

Predicted nutrients for 150g of Apple, raw:
energy_(kcal): 81.08
protein_(g): 0.81
carbohydrate_(g): 17.34
sugars_total_(g): 13.87
fiber_total_dietary_(g): 2.37
total_fat_(g): 1.10
fatty_acids,_total_saturated_(g): 0.29
fatty_acids,_total_monounsaturated_(g): 0.32
fatty_acids,_total_polyunsaturated_(g): 0.30
cholestero